# testing authentication

In [10]:
import base64
import certifi
import configparser
import os
import pprint
import requests
from requests.auth import HTTPBasicAuth
import ssl
import sys
import tempfile

In [11]:
# Get the project root (one level up from notebooks/)
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))

if project_root not in sys.path:
    sys.path.append(project_root)
    
project_root

#read env file
config = configparser.ConfigParser()
config.read(f"{project_root}/env.ini")

['g:\\My Drive\\Data analysis\\Python projects\\energy-usage/env.ini']

In [12]:
# Security software on this machine does SSL inspection and presents its own
# certificate. requests uses certifi's CA bundle, which doesn't include that
# cert, causing handshake failures. Fix: build a combined CA bundle from
# certifi + the Windows certificate store (which does trust the antivirus CA).

pem_certs = [open(certifi.where()).read()]

for store in ("ROOT", "CA"):
    for cert_bytes, encoding, trust in ssl.enum_certificates(store):
        if encoding == "x509_asn":
            b64 = base64.b64encode(cert_bytes).decode()
            pem_certs.append(
                "-----BEGIN CERTIFICATE-----\n"
                + "\n".join(b64[i : i + 64] for i in range(0, len(b64), 64))
                + "\n-----END CERTIFICATE-----"
            )

ca_bundle = tempfile.NamedTemporaryFile(mode="w", suffix=".pem", delete=False)
ca_bundle.write("\n".join(pem_certs))
ca_bundle.close()
print(f"CA bundle written ({len(pem_certs)} certs) → {ca_bundle.name}")

# All requests go through a session that uses this bundle
session = requests.Session()
session.verify = ca_bundle.name

# Sanity check
r = session.get("https://api.octopus.energy/v1/products/", timeout=10)
print("Status:", r.status_code)

CA bundle written (71 certs) → C:\Users\shane\AppData\Local\Temp\tmpv54e4vym.pem
Status: 200


get a token for auth purposes

In [13]:
#now try and get a token
endpoint = "https://api.octopus.energy/v1/graphql/"

mutation = """
mutation obtainToken($apiKey: String!) {
  obtainKrakenToken(input: {APIKey: $apiKey}) {
    token
    refreshToken
    refreshExpiresIn
  }
}
"""

payload = {
    "query": mutation,
    "variables": {
        "apiKey": config["default"]["OCTOPUS_API_KEY"]
    }
}

response = session.post(
    endpoint,
    json=payload,
    headers={"Content-Type": "application/json"},
)

token = response.json()["data"]["obtainKrakenToken"]["token"]
print("Token obtained:", token[:40], "...")

Token obtained: eyJhbGciOiJSUzI1NiIsImlzcyI6Imh0dHBzOi8v ...


Now try a test query

In [14]:
#setup the query
query = """
query {
  viewer {
    accounts {
      number
    }
  }
}
"""

response = session.post(
    endpoint,
    json={"query": query},
    headers={"Authorization": token},
)

print(response.status_code)
print(response.json())

200
{'data': {'viewer': {'accounts': [{'number': 'A-9891ABF5'}]}}}
